In [0]:
%run ./my_packages

In [0]:
# Set `mediatel_date` here if you want to run manually. Otherwise under BAU conditions (i.e. when you don't need to run any past dates) this cell should be skipped via %skip tag.
start = '20260414'
end = '20260416'
mediatel_dates = [
    (datetime.strptime(start, "%Y%m%d") + timedelta(days=i)).strftime("%Y%m%d") for i in range((datetime.strptime(end, "%Y%m%d") - datetime.strptime(start, "%Y%m%d")).days + 1)
    ]

In [0]:
loan_info = spark.sql(
    """
    SELECT DISTINCT
    loanorderno AS Loan_ID,
    business_product,
    disbursed_amount_group,
    tenure_group,
    CASE
        WHEN loan_sequence <= 1 THEN 'Loan 1'
        WHEN loan_sequence BETWEEN 2 AND 5 THEN 'Loan 2-5'
        WHEN loan_sequence BETWEEN 6 AND 12 THEN 'Loan 6-12'
        WHEN loan_sequence >= 13 THEN 'Loan 13+'
        ELSE 'Loan 1'
    END AS sequence_grouping,
    CASE
        WHEN final_credit_score < 600 THEN 'G. <600'
        WHEN final_credit_score >= 600 AND final_credit_score <= 619 THEN 'F. 600-619'
        WHEN final_credit_score >= 620 AND final_credit_score <= 639 THEN 'E. 620-639'
        WHEN final_credit_score >= 640 AND final_credit_score <= 649 THEN 'D. 640-649'
        WHEN final_credit_score >= 650 AND final_credit_score <= 669 THEN 'C. 650-669'
        WHEN final_credit_score >= 670 AND final_credit_score <= 689 THEN 'B. 670-689'
        WHEN final_credit_score >= 690 THEN 'A. 690+'
        ELSE 'G. <600'
    END AS credit_score_group,
    CASE
        WHEN e.credit_policy IN ('ElNido20260317') THEN 'El Nido'
        WHEN e.credit_policy IN ('Davao20251204','Davao20251218','DavaoUnifiedNoRateCapV1') THEN 'Davao'
        WHEN e.credit_policy IN ('DavaoBusinessLoanTest20260123','DavaoDecreasingInstallmentTest20260205','DavaoFirstLoanProductTest20260108','DavaoQrTest20260114','DavaoQrTest20260203','DavaoRepeatLoanProductTest20251216') THEN 'Davao Test'
        WHEN e.credit_policy IN ('CebuV1','CebuV2','CebuV3','CebuV4','CebuV5','CebuV6') THEN 'Cebu'
        WHEN e.credit_policy IN ('CebuHoldout20251127') THEN 'Cebu Test'
        WHEN e.credit_policy IN ('BaguioV1','BaguioV2','BaguioV3','BaguioV4','BaguioV5','BaguioV6','BaguioV7') THEN 'Baguio'
        WHEN e.credit_policy IN ('BaguioHoldout20251127') THEN 'Baguio Test'
        WHEN e.credit_policy IN ('AlaminosV1') THEN 'Alaminos'
        WHEN e.credit_policy IN ('QrControl20260212') THEN 'QR Test'
        ELSE NULL
    END AS credit_policy_group,
    CASE WHEN email_verified = 1 THEN 'Email verified' ELSE 'Email unverified' END AS email_verified
    FROM teams.cashalo.jack_loan_disbursement
    LEFT JOIN (SELECT DISTINCT loanorderno, credit_policy, if_experiment_policy FROM teams.cashalo.jack_loan_eligibility_at_offer_na) AS e USING (loanorderno)
    LEFT JOIN (SELECT DISTINCT user_id_NA, email_verified FROM teams.cashalo.jack_email_verified_na) USING (user_id_NA)
    """
)
loan_info = pl.from_pandas(loan_info.toPandas())

In [0]:
table_name = "teams.cashalo.brett_dial_dashboard"

In [0]:
for mediatel_date in mediatel_dates:
    try:
       # Read excel file
        df = f_LoadLatestMediatel(mediatel_date=mediatel_date, select_cols=select_cols)

        # Create metrics
        df = f_CreateMetrics(df)

        # Add loan info (business product) to the Mediatel extract
        df = (
            df
            .join(loan_info, on='Loan_ID', how='left')
            .filter(pl.col("business_product").is_not_null())
            .filter(pl.col("sequence_grouping").is_not_null())
            )

        # Summarise
        df_summary = f_CreateSummary(df)

        # Create new date columns for aggregating to weekly and monthly views. Add spin targets.
        df_summary = (
            df_summary
            .with_columns(pl.col("call_date").dt.date().alias("call_date"))
            .with_columns(pl.col("call_date").dt.truncate("1w").dt.date().alias("call_week"))
            .with_columns(pl.col("call_date").dt.month_start().dt.date().alias("call_month"))
            .with_columns(pl.when(pl.col("DPD_type") == "Pre-Due").then(7).otherwise(8).alias("HC_Spin_Target"))
            .with_columns(pl.lit(5).alias("VB_Spin_Target"))
        )

        # Convert to pandas dataframe for union with existing summary table
        df_summary = df_summary.to_pandas()

        # Load current summary table, which will be as of t-2
        if spark.catalog.tableExists(table_name):
            print(f'{mediatel_date}: append')
            # Append new data
            spark_df = f_CreateSparkTable(df_summary, table_name=table_name)
            spark_df.write.mode('append').saveAsTable(table_name)
        else:
            print(f'{mediatel_date}: add new')
            # If no table exists save current summary as the table
            spark_df = f_CreateSparkTable(df_summary, table_name=table_name)
            spark_df.write.saveAsTable(table_name)
            
    except Exception as e:
        dbutils.notebook.exit(e)